# Stable Diffusion WebUI & LoRA Train (Public)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

在 Google Colab 免費 GPU 上執行 [Stable Diffusion WebUI (A1111)](https://github.com/AUTOMATIC1111/stable-diffusion-webui)，
透過 Cloudflare Tunnel 暴露 API，讓本地的 **FaceBox** 後端遠端調用 GPU 算力進行圖片生成與 LoRA 訓練。

## 快速開始
1. **Runtime → Change runtime type → T4 GPU**
2. 執行 **Cell 1 (Config)** 設定參數
3. 依序執行 Cell 2–7，等待 Tunnel URL 出現
4. 本地啟動 FaceBox：
   ```bash
   # Windows
   set SD_WEBUI_URL=https://xxx.trycloudflare.com
   cd FaceBox/facebox && python -m backend.main

   # macOS / Linux
   SD_WEBUI_URL=https://xxx.trycloudflare.com python -m backend.main
   ```

## 架構
```
┌─────────────────────────────┐      Cloudflare Tunnel       ┌──────────────────┐
│  Google Colab (T4 GPU)      │◄────────────────────────────► │  本地 FaceBox    │
│  SD WebUI :7860 (--api)     │  https://xxx.trycloudflare.com│  Backend :17494  │
│  + LoRA Training (kohya-ss) │                               │  MCP Server      │
└─────────────────────────────┘                               └──────────────────┘
```

In [ ]:
#@title 1. 設定參數 { run: "auto" }
#@markdown 所有可調參數集中在此，執行前請依需求修改。

# ── 基礎模型 ──────────────────────────────────────────────
#@markdown ### 基礎模型
MODEL_NAME = "realisticVisionV51"  #@param {type:"string"}
MODEL_URL = "https://civitai.com/api/download/models/130072?type=Model&format=SafeTensor&size=pruned&fp=fp16"  #@param {type:"string"}
#@markdown > 預設 Realistic Vision v5.1，適合人像。可替換為任何 SD 1.5 模型的直連下載 URL。

# ── Google Drive 持久化 ──────────────────────────────────
#@markdown ### Google Drive
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SD_WebUI_Colab"  #@param {type:"string"}
#@markdown > 開啟後模型和 LoRA 會存到 Drive，下次啟動免重新下載。

# ── SD WebUI 設定 ──────────────────────────────────────────
#@markdown ### SD WebUI
WEBUI_PORT = 7860  #@param {type:"integer"}
WEBUI_ARGS = "--api --opt-sdp-attention --no-half-vae --listen --enable-insecure-extension-access --skip-python-version-check"  #@param {type:"string"}

# ── LoRA 訓練預設（與 FaceBox config.py 對齊）─────────────
#@markdown ### LoRA 訓練
LORA_RESOLUTION = 384  #@param {type:"integer"}
LORA_TRAIN_STEPS = 1500  #@param {type:"integer"}
LORA_LEARNING_RATE = 1e-4  #@param {type:"number"}
LORA_NETWORK_DIM = 16  #@param {type:"integer"}
LORA_NETWORK_ALPHA = 8  #@param {type:"integer"}
LORA_TRIGGER_WORD = "sks person"  #@param {type:"string"}

# ── 衍生路徑（不需修改）─────────────────────────────────────
import os

WEBUI_DIR = "/content/stable-diffusion-webui"
KOHYA_DIR = "/content/sd-scripts"

if USE_GOOGLE_DRIVE:
    MODELS_DIR = f"{DRIVE_ROOT}/models"
    LORA_OUTPUT_DIR = f"{DRIVE_ROOT}/lora_output"
    TRAIN_DATA_DIR = f"{DRIVE_ROOT}/train_data"
else:
    MODELS_DIR = f"{WEBUI_DIR}/models/Stable-diffusion"
    LORA_OUTPUT_DIR = "/content/lora_output"
    TRAIN_DATA_DIR = "/content/train_data"

for d in [MODELS_DIR, LORA_OUTPUT_DIR, TRAIN_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ 參數載入完成")
print(f"   模型: {MODEL_NAME}")
print(f"   Drive 持久化: {'開啟' if USE_GOOGLE_DRIVE else '關閉'}")
print(f"   WebUI Port: {WEBUI_PORT}")

✅ 參數載入完成
   模型: realisticVisionV51
   Drive 持久化: 開啟
   WebUI Port: 7860


In [3]:
#@title 2. 環境檢查（GPU + Python）
import subprocess, sys

# GPU 檢查
print("── GPU 資訊 ──")
gpu_result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"], capture_output=True, text=True)
if gpu_result.returncode != 0:
    print("❌ 未偵測到 GPU！請至 Runtime → Change runtime type → T4 GPU")
    raise SystemExit("No GPU detected")

gpu_info = gpu_result.stdout.strip()
print(f"   {gpu_info}")

# Python 版本檢查
py_ver = sys.version_info
print(f"\n── Python ──")
print(f"   {sys.version}")
if py_ver.major != 3 or py_ver.minor not in (10, 11, 12):
    print(f"   ⚠️  建議使用 Python 3.10–3.12，當前 {py_ver.major}.{py_ver.minor} 可能有相容性問題")
else:
    print(f"   ✅ Python 版本相容")

# 磁碟空間
print(f"\n── 磁碟空間 ──")
!df -h /content | tail -1 | awk '{print "   可用:", $4, "/", $2}'

print("\n✅ 環境檢查完成")

── GPU 資訊 ──
   Tesla T4, 15360 MiB, 580.82.07

── Python ──
   3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
   ✅ Python 版本相容

── 磁碟空間 ──
   可用: 70G / 113G

✅ 環境檢查完成


In [4]:
#@title 3. 掛載 Google Drive（選用）
if USE_GOOGLE_DRIVE:
    import subprocess
    # 清理殘留掛載（runtime 重啟後可能有殘留檔案）
    subprocess.run(["fusermount", "-u", "/content/drive"], capture_output=True)
    if os.path.exists("/content/drive"):
        import shutil
        shutil.rmtree("/content/drive", ignore_errors=True)

    from google.colab import drive
    drive.mount("/content/drive")

    for d in [MODELS_DIR, LORA_OUTPUT_DIR, TRAIN_DATA_DIR]:
        os.makedirs(d, exist_ok=True)

    print(f"✅ Google Drive 已掛載")
    print(f"   模型目錄:  {MODELS_DIR}")
    print(f"   LoRA 輸出: {LORA_OUTPUT_DIR}")
    print(f"   訓練資料:  {TRAIN_DATA_DIR}")
else:
    print("⏭️  跳過 Drive 掛載（USE_GOOGLE_DRIVE = False）")

Mounted at /content/drive
✅ Google Drive 已掛載
   模型目錄:  /content/drive/MyDrive/FaceBox/models
   LoRA 輸出: /content/drive/MyDrive/FaceBox/lora_output
   訓練資料:  /content/drive/MyDrive/FaceBox/train_data


In [5]:
#@title 4. 安裝 SD WebUI (A1111) + 下載模型
import os, shutil

# ── Clone SD WebUI ──
if not os.path.exists(f"{WEBUI_DIR}/launch.py"):
    print("📦 正在安裝 Stable Diffusion WebUI...")
    !git clone --depth 1 https://github.com/AUTOMATIC1111/stable-diffusion-webui.git {WEBUI_DIR}
    print("✅ SD WebUI 安裝完成")
else:
    print("✅ SD WebUI 已存在，跳過安裝")

# ── 下載基礎模型 ──
model_path = f"{MODELS_DIR}/{MODEL_NAME}.safetensors"
webui_model_path = f"{WEBUI_DIR}/models/Stable-diffusion/{MODEL_NAME}.safetensors"

if os.path.exists(model_path):
    print(f"✅ 模型已存在: {model_path}")
elif MODEL_URL:
    print(f"📥 正在下載 {MODEL_NAME}...")
    !wget -q --show-progress -O {model_path} "{MODEL_URL}"
    if os.path.exists(model_path) and os.path.getsize(model_path) > 1_000_000:
        print(f"✅ 模型下載完成 ({os.path.getsize(model_path) / 1e9:.1f} GB)")
    else:
        print("❌ 模型下載失敗，請檢查 MODEL_URL")
        if os.path.exists(model_path):
            os.remove(model_path)
else:
    print("⏭️  未設定 MODEL_URL，跳過模型下載")

# ── 建立符號連結（Drive → WebUI）──
if USE_GOOGLE_DRIVE and os.path.exists(model_path):
    os.makedirs(os.path.dirname(webui_model_path), exist_ok=True)
    if not os.path.exists(webui_model_path):
        os.symlink(model_path, webui_model_path)
        print(f"🔗 已建立符號連結: Drive → WebUI models/")

# ── LoRA 目錄連結 ──
webui_lora_dir = f"{WEBUI_DIR}/models/Lora"
os.makedirs(webui_lora_dir, exist_ok=True)
if USE_GOOGLE_DRIVE:
    for f in os.listdir(LORA_OUTPUT_DIR):
        if f.endswith(".safetensors"):
            src = os.path.join(LORA_OUTPUT_DIR, f)
            dst = os.path.join(webui_lora_dir, f)
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"🔗 已連結 LoRA: {f}")

print("\n✅ 模型準備完成")
!ls -lh {WEBUI_DIR}/models/Stable-diffusion/ 2>/dev/null || true

📦 正在安裝 Stable Diffusion WebUI...
Cloning into '/content/stable-diffusion-webui'...
remote: Enumerating objects: 363, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (327/327), done.
remote: Total 363 (delta 14), reused 243 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (363/363), 1.80 MiB | 22.00 MiB/s, done.
Resolving deltas: 100% (14/14), done.
✅ SD WebUI 安裝完成
✅ 模型已存在: /content/drive/MyDrive/FaceBox/models/realisticVisionV51.safetensors
🔗 已建立符號連結: Drive → WebUI models/

✅ 模型準備完成
total 4.0K
-rw-r--r-- 1 root root  0 Mar 20 07:00 'Put Stable Diffusion checkpoints here.txt'
lrwxrwxrwx 1 root root 68 Mar 20 07:00  realisticVisionV51.safetensors -> /content/drive/MyDrive/FaceBox/models/realisticVisionV51.safetensors


In [15]:
#@title 5. 安裝 Cloudflare Tunnel
import subprocess, shutil

if shutil.which("cloudflared"):
    result = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
    print(f"✅ cloudflared 已安裝: {result.stdout.strip() or result.stderr.strip()}")
else:
    print("📦 正在安裝 cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    result = subprocess.run(["cloudflared", "--version"], capture_output=True, text=True)
    print(f"✅ cloudflared 安裝完成: {result.stdout.strip() or result.stderr.strip()}")

📦 正在安裝 cloudflared...
✅ cloudflared 安裝完成: cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


In [21]:
#@title 6. 啟動 SD WebUI + Tunnel 🚀
#@markdown 執行後等待出現 Tunnel URL，複製到本地 FaceBox 使用。
#@markdown
#@markdown > ⏱️ 首次啟動約需 3-5 分鐘（安裝依賴），後續約 1 分鐘。

import subprocess, threading, re, time, urllib.request, json, os, shutil

# ── 修復 Python 3.12 相容性問題 ──
# transformers 4.30.2 的 tokenizers 沒有 cp312 wheel
req_file = f"{WEBUI_DIR}/requirements_versions.txt"
if os.path.exists(req_file):
    with open(req_file, "r") as f:
        content = f.read()
    if "transformers==4.30.2" in content:
        content = content.replace("transformers==4.30.2", "transformers==4.38.2")
        with open(req_file, "w") as f:
            f.write(content)
        print("🔧 已修復 transformers 版本（4.30.2 → 4.38.2）")

# Colab 預裝的 wandb 版本有 bug，A1111 不需要
!pip uninstall -y wandb > /dev/null 2>&1
print("🔧 已移除有問題的 wandb")

# ── 修復 Stability-AI repo（已被設為私有，改用 mirror）──
sd_repo_dir = f"{WEBUI_DIR}/repositories/stable-diffusion-stability-ai"
if not os.path.exists(sd_repo_dir):
    print("📦 正在克隆 Stable Diffusion repo...")
    os.makedirs(f"{WEBUI_DIR}/repositories", exist_ok=True)
    !git clone https://github.com/camenduru/stablediffusion.git {sd_repo_dir}
    print("✅ Stable Diffusion repo 已就緒")

# ── 修復 generative-models repo（需要完整歷史才能 checkout 指定 commit）──
gen_repo_dir = f"{WEBUI_DIR}/repositories/generative-models"
gen_commit = "45c443b316737a4ab6e40413d7794a7f5657c19f"
if os.path.exists(gen_repo_dir):
    ret = subprocess.run(["git", "-C", gen_repo_dir, "cat-file", "-t", gen_commit],
                         capture_output=True)
    if ret.returncode != 0:
        print("⚠️  generative-models repo 缺少所需 commit，重新克隆...")
        shutil.rmtree(gen_repo_dir, ignore_errors=True)
if not os.path.exists(gen_repo_dir):
    print("📦 正在克隆 generative-models repo（需要完整歷史）...")
    !git clone https://github.com/camenduru/generative-models.git {gen_repo_dir}
    print("✅ generative-models repo 已就緒")

_tunnel_url = None

def _run_tunnel():
    """背景啟動 cloudflared，偵測到 URL 後印出。"""
    global _tunnel_url
    time.sleep(20)
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{WEBUI_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    for line in proc.stderr:
        m = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line)
        if m:
            _tunnel_url = m.group(1)
            print("\n" + "=" * 64)
            print(f"  🌐 Tunnel URL: {_tunnel_url}")
            print(f"" )
            print(f"  本地 FaceBox 連線指令：")
            print(f"  [Windows]  set SD_WEBUI_URL={_tunnel_url}")
            print(f"  [Linux]    export SD_WEBUI_URL={_tunnel_url}")
            print("=" * 64 + "\n")
            break

def _health_check():
    """持續檢查 SD WebUI 是否就緒。"""
    while True:
        time.sleep(10)
        try:
            req = urllib.request.urlopen(f"http://127.0.0.1:{WEBUI_PORT}/sdapi/v1/sd-models", timeout=5)
            data = json.loads(req.read())
            if data:
                print(f"✅ SD WebUI 就緒！已載入 {len(data)} 個模型")
                return
        except Exception:
            pass

threading.Thread(target=_run_tunnel, daemon=True).start()
threading.Thread(target=_health_check, daemon=True).start()

# 啟動 SD WebUI（此 cell 會持續執行，不會結束）
!cd {WEBUI_DIR} && python launch.py \
    {WEBUI_ARGS} \
    --port {WEBUI_PORT}

🔧 已移除有問題的 wandb
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Version: v1.10.1
Commit hash: 82a973c04367123ae98bd9abdf80d9eda9b910e2
Launching Web UI with arguments: --api --opt-sdp-attention --no-half-vae --listen --enable-insecure-extension-access --skip-python-version-check --port 7860
2026-03-20 07:42:07.141924: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773992527.164997   14210 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773992527.172119   14210 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773992527.190604   14210 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid 

---

## LoRA 訓練（選用）

以下 Cell 用於在 Colab GPU 上訓練人臉 LoRA。訓練完成後可直接用於 SD WebUI 生成，
或下載回本地讓 FaceBox 使用。

**流程：** 安裝 kohya-ss → 上傳訓練照片 → 訓練 LoRA → 部署/下載

In [6]:
#@title 7. 安裝 kohya-ss/sd-scripts（LoRA 訓練框架）
import os

if not os.path.exists(f"{KOHYA_DIR}/train_network.py"):
    print("📦 正在安裝 kohya-ss...")
    !pip install -q accelerate safetensors transformers diffusers lion-pytorch prodigyopt bitsandbytes
    !git clone --depth 1 https://github.com/kohya-ss/sd-scripts.git {KOHYA_DIR}
    !cd {KOHYA_DIR} && pip install -q -r requirements.txt
    print("✅ kohya-ss 安裝完成")
else:
    print("✅ kohya-ss 已存在，跳過安裝")

📦 正在安裝 kohya-ss...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.7 MB/s eta 0:00:00:00:0100:01
Cloning into '/content/sd-scripts'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (226/226), done.
remote: Total 247 (delta 41), reused 100 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 11.67 MiB | 22.98 MiB/s, done.
Resolving deltas: 100% (41/41), done.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 116.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
#@title 8. 上傳訓練照片
#@markdown 選擇照片來源：從本機上傳或從 Google Drive 讀取。
#@markdown
#@markdown 建議 **10-20 張**，包含正面、側面、不同光線的清晰照片。

PHOTO_SOURCE = "drive"  #@param ["upload", "drive"]
DRIVE_PHOTOS_PATH = "/content/drive/MyDrive/SD_WebUI_Colab/photos"  #@param {type:"string"}
#@markdown > `upload`: 從本機上傳 / `drive`: 從 `DRIVE_PHOTOS_PATH` 複製照片
#@markdown >
#@markdown > 使用 `drive` 模式前，請先將照片放到 Drive 的指定資料夾。

import os, glob, shutil
from IPython.display import display, HTML

# kohya-ss 格式：{repeats}_{trigger_word}
repeats = 10
instance_dir = os.path.join(TRAIN_DATA_DIR, f"{repeats}_{LORA_TRIGGER_WORD}")
os.makedirs(instance_dir, exist_ok=True)

if PHOTO_SOURCE == "upload":
    from google.colab import files
    print("請上傳訓練照片（支援 jpg/png/webp）：")
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(instance_dir, name), "wb") as f:
            f.write(data)
    print(f"\n✅ 已上傳 {len(uploaded)} 張照片")
else:
    # 從 Google Drive 複製照片
    print(f"📂 從 Drive 讀取: {DRIVE_PHOTOS_PATH}")
    if not os.path.exists(DRIVE_PHOTOS_PATH):
        print(f"❌ 找不到資料夾: {DRIVE_PHOTOS_PATH}")
        print(f"   請先將照片上傳到 Google Drive 的該路徑")
    else:
        extensions = (".jpg", ".jpeg", ".png", ".webp")
        copied = 0
        for f in os.listdir(DRIVE_PHOTOS_PATH):
            if f.lower().endswith(extensions):
                src = os.path.join(DRIVE_PHOTOS_PATH, f)
                dst = os.path.join(instance_dir, f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    copied += 1
        print(f"✅ 已從 Drive 複製 {copied} 張新照片")

# 統計
photos = glob.glob(os.path.join(instance_dir, "*.*"))
photo_count = len([p for p in photos if p.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))])

print(f"\n📊 訓練照片統計：")
print(f"   目錄: {instance_dir}")
print(f"   照片數: {photo_count}")
print(f"   重複次數: {repeats}")
print(f"   觸發詞: {LORA_TRIGGER_WORD}")

if photo_count < 5:
    print(f"\n⚠️  照片數量偏少（{photo_count} 張），建議至少 10 張以獲得較好效果")
elif photo_count > 30:
    print(f"\n⚠️  照片數量偏多（{photo_count} 張），可能需要增加訓練步數")

📂 從 Drive 讀取: /content/drive/MyDrive/FaceBox/photos
❌ 找不到資料夾: /content/drive/MyDrive/FaceBox/photos
   請先將照片上傳到 Google Drive 的該路徑

📊 訓練照片統計：
   目錄: /content/drive/MyDrive/FaceBox/train_data/10_sks person
   照片數: 14
   重複次數: 10
   觸發詞: sks person


In [11]:
#@title 9. 開始 LoRA 訓練
#@markdown ⏱️ T4 GPU 約需 20-40 分鐘（依步數而定）

LORA_NAME = "facebox_lora"  #@param {type:"string"}

import os, time

# 確認模型路徑
base_model = f"{WEBUI_DIR}/models/Stable-diffusion/{MODEL_NAME}.safetensors"
if not os.path.exists(base_model):
    base_model = f"{MODELS_DIR}/{MODEL_NAME}.safetensors"
    if not os.path.exists(base_model):
        raise FileNotFoundError("找不到基礎模型，請先執行 Cell 4")

print("── LoRA 訓練設定 ──")
print(f"   基礎模型:    {os.path.basename(base_model)}")
print(f"   LoRA 名稱:   {LORA_NAME}")
print(f"   解析度:      {LORA_RESOLUTION}")
print(f"   訓練步數:    {LORA_TRAIN_STEPS}")
print(f"   學習率:      {LORA_LEARNING_RATE}")
print(f"   Network Dim: {LORA_NETWORK_DIM}")
print(f"   輸出目錄:    {LORA_OUTPUT_DIR}")
print()

start_time = time.time()

!cd {KOHYA_DIR} && accelerate launch \
    --num_cpu_threads_per_process 2 \
    train_network.py \
    --pretrained_model_name_or_path="{base_model}" \
    --train_data_dir="{TRAIN_DATA_DIR}" \
    --output_dir="{LORA_OUTPUT_DIR}" \
    --output_name="{LORA_NAME}" \
    --resolution={LORA_RESOLUTION} \
    --train_batch_size=1 \
    --max_train_steps={LORA_TRAIN_STEPS} \
    --learning_rate={LORA_LEARNING_RATE} \
    --network_module=networks.lora \
    --network_dim={LORA_NETWORK_DIM} \
    --network_alpha={LORA_NETWORK_ALPHA} \
    --mixed_precision=fp16 \
    --save_precision=fp16 \
    --optimizer_type=AdamW8bit \
    --sdpa \
    --cache_latents \
    --save_every_n_steps=500

elapsed = time.time() - start_time
lora_file = os.path.join(LORA_OUTPUT_DIR, f"{LORA_NAME}.safetensors")

if os.path.exists(lora_file):
    size_mb = os.path.getsize(lora_file) / 1e6
    print(f"\n✅ LoRA 訓練完成！")
    print(f"   耗時: {elapsed/60:.1f} 分鐘")
    print(f"   檔案: {lora_file}")
    print(f"   大小: {size_mb:.1f} MB")
else:
    print(f"\n❌ 訓練可能失敗，請檢查上方日誌")

── LoRA 訓練設定 ──
   基礎模型:    realisticVisionV51.safetensors
   LoRA 名稱:   facebox_lora
   解析度:      384
   訓練步數:    1500
   學習率:      0.0001
   Network Dim: 16
   輸出目錄:    /content/drive/MyDrive/FaceBox/lora_output

	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-03-20 07:07:28.476493: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773990448.498029    4019 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773990448.505747    4019 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to re

In [13]:
#@title 10. 部署 LoRA 到 SD WebUI + 下載
#@markdown 將訓練好的 LoRA 部署到執行中的 SD WebUI，並可下載到本機。

import os, shutil, urllib.request

lora_file = os.path.join(LORA_OUTPUT_DIR, f"{LORA_NAME}.safetensors")
webui_lora_dir = f"{WEBUI_DIR}/models/Lora"

if not os.path.exists(lora_file):
    print(f"❌ 找不到 LoRA: {lora_file}")
    print(f"   請先執行 Cell 9 進行訓練")
else:
    # 複製到 SD WebUI Lora 目錄
    dst = os.path.join(webui_lora_dir, f"{LORA_NAME}.safetensors")
    if not os.path.exists(dst):
        shutil.copy2(lora_file, dst)
        print(f"✅ 已複製到 SD WebUI: {dst}")
    else:
        print(f"✅ SD WebUI 已有此 LoRA")

    # 通知 SD WebUI 刷新 LoRA 清單
    try:
        req = urllib.request.Request(
            f"http://127.0.0.1:{WEBUI_PORT}/sdapi/v1/refresh-loras",
            method="POST",
            headers={"Content-Type": "application/json"},
            data=b""
        )
        urllib.request.urlopen(req, timeout=10)
        print(f"✅ SD WebUI LoRA 清單已刷新")
    except Exception as e:
        print(f"⚠️  無法刷新 LoRA 清單（SD WebUI 可能未執行）: {e}")

    # 檔案資訊
    size_mb = os.path.getsize(lora_file) / 1e6
    print(f"\n📦 LoRA 檔案資訊:")
    print(f"   名稱: {LORA_NAME}.safetensors")
    print(f"   大小: {size_mb:.1f} MB")
    print(f"   觸發詞: {LORA_TRIGGER_WORD}")

    # 下載到本機
    DOWNLOAD_TO_LOCAL = True  #@param {type:"boolean"}
    if DOWNLOAD_TO_LOCAL:
        from google.colab import files
        files.download(lora_file)
        print(f"\n📥 正在下載到本機...")
        print(f"   下載後放到 FaceBox/facebox/data/lora_models/ 即可使用")

✅ SD WebUI 已有此 LoRA
⚠️  無法刷新 LoRA 清單（SD WebUI 可能未執行）: <urlopen error [Errno 111] Connection refused>

📦 LoRA 檔案資訊:
   名稱: facebox_lora.safetensors
   大小: 19.0 MB
   觸發詞: sks person


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 正在下載到本機...
   下載後放到 FaceBox/facebox/data/lora_models/ 即可使用


---

## 疑難排解

| 問題 | 解法 |
|------|------|
| **Tunnel URL 沒出現** | 等待 SD WebUI 完全啟動（首次約 3-5 分鐘），URL 會自動印出 |
| **FaceBox 連不上** | 確認 `SD_WEBUI_URL` 設定正確，且 Colab Cell 6 仍在執行中 |
| **Out of Memory** | 降低生成解析度，或使用 `--medvram` 加入 WEBUI_ARGS |
| **模型下載失敗** | CivitAI 可能限流，稍後重試或手動上傳模型到 Drive |
| **LoRA 訓練 OOM** | 降低 `LORA_RESOLUTION` 至 384，或 `LORA_NETWORK_DIM` 至 16 |
| **Colab 斷線** | 開啟 Drive 持久化，重連後只需重新執行 Cell 4-6 |

## 授權

MIT License — 詳見 [LICENSE](../LICENSE)